<div style="background-color:#DFC7F2; padding:15px; margin:15px;">


**Центр непрерывного образования НИУ ВШЭ**

## Программа «Python для автоматизации и анализа данных»

### Занятие 10. Часть 3. Выгрузка интернет-данных в табличном формате

*Авторы: Алла Тамбовцева, Маргарита Бурова, НИУ ВШЭ*

</div>

## Веб-скрейпинг

Мы уже немного освоились со структурой html-файлов, теперь попробуем выгрузить информацию с реального сайта — [nplus1.ru](https://nplus1.ru/).

**Наша задача:** выгрузить недавние новости в датафрейм `pandas`, чтобы потом сохранить всё в csv-файл.

Сначала скачаем весь html-код страницы и сохраним его в отдельной переменной. Для этого нам понадобится библиотека `requests`. Импортируем её:

In [1]:
import requests

Сохраним ссылку на главную страницу сайта в переменную `url` для удобства и выгрузим страницу. (Разумеется, это будет работать только при подключении к интернету. Если соединения нет, Python выдаст `NewConnectionError`.)

In [2]:
url = 'https://nplus1.ru/' # сохраняем ссылку
page = requests.get(url) # загружаем страницу по ссылке

Если мы просто посмотрим на объект, ничего особенного не увидим:

In [3]:
page # ответ 200 - страница загружена

<Response [200]>

In [4]:
# page.text

Импортируем функцию `BeautifulSoup` из библиотеки `bs4` (от *beautifulsoup4*) и превратим html-код страницы `page` из текста в объект, по которому удобно искать.

In [5]:
from bs4 import BeautifulSoup # не спрашивайте, почему BeautifulSoup

In [46]:
page.text

'<!DOCTYPE html>\n<html class="bg-light-gray" prefix="og: http://ogp.me/ns#" lang="ru">\n<head>\n  <meta charset="utf-8">\n  <meta name="viewport" content="width=device-width, initial-scale=1">\n  <title>N + 1 — главное издание о науке, технике и технологиях</title>\n  <link rel="preload" href="https://staticn1.nplus1.ru/fonts/AeonikPro/AeonikPro-Regular.woff2" as="font"\n        type="font/woff2" crossorigin/>\n  <link rel="preload" href="https://staticn1.nplus1.ru/fonts/Spectral/Spectral-Regular.woff" as="font"\n        type="font/woff2" crossorigin/>\n  <link href="/front-build/immutable/css/main.6cf2cea474c94152aefa.css" rel="stylesheet">\n  <link href="/front-build/immutable/css/app.588032dc28c6038f3dce.css" rel="stylesheet">\n  \n\n  <link rel="apple-touch-icon" sizes="180x180" href="/apple-touch-icon.png">\n  <link rel="icon" type="image/png" sizes="32x32" href="/favicon-32x32.png">\n  <link rel="icon" type="image/png" sizes="16x16" href="/favicon-16x16.png">\n  <link rel="manif

In [6]:
soup = BeautifulSoup(page.text, 'html')

Если вывести `soup` на экран, мы увидим то же самое, что в режиме разработчика или в режиме просмотра исходного кода (`view-source` через *Ctrl+U* в Google Chrome).

In [7]:
# soup

Смотреть так не очень удобно. «Причешем» наш `soup` — воспользуемся методом `.prettify()` в сочетании с функцией `print()`.

In [8]:
# print(soup.prettify())

В такой выдаче ориентироваться гораздо удобнее (но при желании то же самое можно увидеть в браузере, на большом экране).

Чтобы загрузить все новости с главной страницы сайта, нужно собрать все ссылки на страницы с новостями. Ссылки в html-файле всегда заключены в тег `<a></a>` и имеют атрибут `href`. Посмотрим на фрагменты кода, соответствующие всем ссылкам на главной странице сайта:

In [9]:
for link in soup.find_all('a'):
    print(link.get('href'))
    # break

/
/search
/subscription?utm_source=nplus&utm_medium=nybanner&utm_content=general
https://nplus1.ru/about
https://nplus1.ru/difficult
https://nplus1.ru/adv
https://nplus1.ru/blog/2022/04/01/samotek
https://nplus1.ru/search?tags=946
https://nplus1.ru/search?tags=869
https://nplus1.ru/search?tags=874
https://nplus1.ru/search?tags=880
https://nplus1.ru/search?tags=768
https://nplus1.ru/search?tags=890
https://nplus1.ru/search?tags=871
https://nplus1.ru/search?tags=876
https://nplus1.ru/search?tags=775
https://nplus1.ru/search?tags=767
https://nplus1.ru/search?tags=771
https://nplus1.ru/search?tags=772
https://nplus1.ru/search?tags=778
https://nplus1.ru/search?tags=824
https://nplus1.ru/search?tags=908
https://nplus1.ru/search?tags=812
https://t.me/nplusone
https://vk.com/nplusone
https://ok.ru/nplus1
https://twitter.com/nplusodin
https://nplus1.ru/about
https://nplus1.ru/difficult
https://nplus1.ru/adv
https://nplus1.ru/blog/2022/04/01/samotek
https://nplus1.ru/search?tags=946
https://nplu

Ссылок много. Но нам нужны только новости — ссылки, в которых есть `/news`. Добавим условие: будем выбирать только такие ссылки. Создадим список `urls` и добавим в него только ссылки, которые удовлетворяют этому условию.

In [10]:
# urls = []

# for link in soup.find_all('a'):
#     if '/news' in link.get('href'):
#         urls.append('https://nplus1.ru' + link.get('href'))

urls = [link.get('href')
        for link in soup.find_all('a')
        if link.get('href') and 'https://nplus1.ru/news/2026' in link.get('href')]

Раньше ссылки на сайте были относительными: неполными, без начала (названия сайта), и их приходилось склеивать с https://nplus1.ru (так сделано в закомментированном варианте выше). Сейчас ссылки на новости абсолютные, поэтому мы просто отбираем те, что начинаются с `https://nplus1.ru/news/2026`.

In [11]:
urls[:5]

['https://nplus1.ru/news/2026/11/12/social-use-of-meow',
 'https://nplus1.ru/news/2026/11/13/alcedo-atthis-nestlings',
 'https://nplus1.ru/news/2026/11/13/chimpanzee-unvoiced-calls',
 'https://nplus1.ru/news/2026/11/13/brittle-solids-splitting',
 'https://nplus1.ru/news/2026/11/13/wildfire-smoke-asthma']

Теперь наша задача сводится к следующему: изучить одну страницу с новостью, научиться доставать из неё текст и всю необходимую информацию, а затем применить весь набор действий к каждой ссылке из `urls` в цикле. Посмотрим на первую новость (у вас она может быть другой — новости обновляются).

In [12]:
url0 = urls[0]

page0 = requests.get(url0)
soup0 = BeautifulSoup(page0.text, 'html')

In [13]:
url0

'https://nplus1.ru/news/2026/11/12/social-use-of-meow'

В коде каждой страницы с новостью есть метаинформация: дата, имя автора и прочее. Такая информация заключена в тег `<meta></meta>`. Посмотрим:

In [14]:
soup0.find_all('meta')

[<meta charset="utf-8"/>,
 <meta content="width=device-width, initial-scale=1" name="viewport"/>,
 <meta content="#f26e40" name="msapplication-TileColor"/>,
 <meta content="#ffffff" name="theme-color"/>,
 <meta content="8c90b02c84ac3b72" name="yandex-verification"/>,
 <meta content="b419949322895fc9106e24ed01be58ac" name="pmail-verification"/>,
 <meta content="N + 1 — главное издание о науке, технике и технологиях" name="description"/>,
 <meta content="N + 1 — главное издание о науке, технике и технологиях" property="og:site_name"/>,
 <meta content="Кошки замяукали в присутствии хозяина и при виде недоступной еды" property="og:title"/>,
 <meta content="https://minio.nplus1.ru/app-images/media-revisions/1153440/a7fca996-096f-4c6e-9f78-9415c88842bb/6af5303161e4a_cover_share.jpeg" property="og:image"/>,
 <meta content="https://nplus1.ru/news/2026/11/12/social-use-of-meow" property="og:url"/>,
 <meta content="N + 1 — главное издание о науке, технике и технологиях" property="og:description"

Из этого списка нам нужны части с именем автора, датой, заголовком и кратким описанием. Воспользуемся поиском по атрибуту `name`. Передадим функции `find_all()` в качестве аргумента словарь с именем и значением атрибута:

In [15]:
soup0.find_all('meta', {'name' : 'author'}) # например, автор

[<meta content="Катерина Петрова" name="author"/>]

Теперь выберем единственный элемент полученного списка (с индексом 0):

In [16]:
soup0.find_all('meta', {'name' : 'author'})[0]

<meta content="Катерина Петрова" name="author"/>

Нам нужно вытащить из этого объекта `content` — имя автора. Посмотрим на атрибуты:

In [17]:
soup0.find_all('meta', {'name' : 'author'})[0].attrs

{'name': 'author', 'content': 'Катерина Петрова'}

Как получить отсюда `content`? Очень просто, ведь это словарь! А доставать из словаря значение по ключу мы умеем. То же самое делает и метод `.get()`:

In [18]:
author = soup0.find_all('meta', {'name' : 'author'})[0].get('content')
author

'Катерина Петрова'

Аналогичным образом извлечём дату, заголовок и описание.

In [19]:
soup0.find_all('meta', {'property' : 'og:title'})[0].get('content')

'Кошки замяукали в\xa0присутствии хозяина и\xa0при виде недоступной еды'

In [20]:
date = soup0.find_all('meta', {'itemprop' : 'datePublished'})[0].get('content')
title = soup0.find_all('meta', {'property' : 'og:title'})[0].get('content')
description = soup0.find_all('h1', {'class':'text-34 md:text-42 xl:text-52 break-words'})[0].get_text().strip() \
+ '\n' + soup0.find_all('p', {'class':"text-36 md:text-44 xl:text-54 font-spectral text-main-gray mb-6"})[0].get_text().strip()

In [21]:
description

'Кошки замяукали в\xa0присутствии хозяина и\xa0при виде недоступной еды\nВ\xa0одиночестве они мяукали реже'

Осталось вытащить рубрики и сложность текста. Если мы посмотрим на исходный код страницы, то увидим, что нужная нам информация находится в теге `<div>` с тегами, внутри тегов `<span></span>`:

In [47]:
new_data = soup0.find_all('div', {'class':"flex flex-wrap lg:mb-10 gap-2 text-tags xl:pr-9"})[0].find_all('span')[1:]

time_published = new_data[0].get_text()
date_published = new_data[1].get_text()
complexity = new_data[2].get_text()
themes = ', '.join(i.get_text() for i in new_data[3:])

In [23]:
new_data

[<span class="group-hover:text-main transition-colors duration-75">13:07</span>,
 <span class="group-hover:text-main transition-colors duration-75">12.11.26</span>,
 <span class="group-hover:text-main transition-colors duration-75">1.5</span>,
 <span class="group-hover:text-main transition-colors duration-75">Зоология</span>]

Теперь соберём сам текст новости — он лежит в абзацах `<p>`:

In [48]:
paragraphs = soup0.find_all('p', {'class': 'mb-6'})[1:]

In [49]:
paragraphs

[<p class="mb-6">Проанализировав динамику рифтовой зоны Калифорнийского залива и процессы, связанные с перемещением Тихоокеанской плиты, геофизики предположили, что быстрый рифтогенез, происходивший около шести–восьми миллионов лет назад, обусловлен возникновением системы двойной субдукции в западной части Тихого океана. Как <a href="http://www.science.org/doi/10.1126/science.aef5187" rel="noreferrer noopener" target="_blank">говорится</a> в опубликованном в <em>Science</em> исследовании, ее образование могло привести к распространению растягивающих напряжений, ускоривших рифтинг на противоположном краю обширной Тихоокеанской плиты.</p>,
 <p class="mb-6">Предпосылки формирования рифтовой зоны в районе современного Калифорнийского залива появились около 25–30 миллионов лет назад после длительной субдукции древней океанической Фараллонской плиты под Северо-Американскую плиту. На основе геофизических данных и химического анализа магматических пород ученые пришли к выводу, что в это время 

In [50]:
links = set()
for p in paragraphs:
    links |= {i.get('href') for i in p.find_all('a')}
links = ', '.join(links)

In [51]:
full_text = '\n'.join([p.get_text() for p in paragraphs])

Не прошло и двух пар, как мы со всем разобрались :) Теперь осталось совсем чуть-чуть: написать функцию, которая выполняет все проделанные нами действия, и применить её в цикле ко всем ссылкам из списка `urls`. Напишем! Аргументом функции будет ссылка на новость, а возвращать она будет текст новости и всю необходимую информацию (дату, автора, сложность и прочее). Скопируем все строки кода выше.

In [52]:
def GetNews(url0):
    """
    Возвращает кортеж со значениями
    url0, date_published, time_published, author, description, title, complexity, themes, links, full_text
    Параметры:

    url0 — ссылка на новость (строка)
    """
    page0 = requests.get(url0)
    soup0 = BeautifulSoup(page0.text, 'lxml')

    author = soup0.find_all('meta', {'name' : 'author'})[0].get('content')
    date = soup0.find_all('meta', {'itemprop' : 'datePublished'})[0].get('content')
    title = soup0.find_all('meta', {'property' : 'og:title'})[0].get('content')
    description = soup0.find_all('h1', {'class':'text-34 md:text-42 xl:text-52 break-words'})[0].get_text().strip() \
    + '\n' + soup0.find_all('p', {'class':"text-36 md:text-44 xl:text-54 font-spectral text-main-gray mb-6"})[0].get_text().strip()

    new_data = soup0.find_all('div', {'class':"flex flex-wrap lg:mb-10 gap-2 text-tags xl:pr-9"})[0].find_all('span')[1:]

    time_published = new_data[0].get_text()
    date_published = new_data[1].get_text()
    complexity = new_data[2].get_text()
    themes = ', '.join(i.get_text() for i in new_data[3:])

    paragraphs = soup0.find_all('p', {'class': 'mb-6'})[1:]
    links = set()
    for p in paragraphs:
        links |= {i.get('href') for i in p.find_all('a')}
    links -= {None} # если вдруг попалась ссылка без href
    links = ', '.join(links)
    full_text = '\n'.join([p.get_text() for p in paragraphs])

    return url0, date_published, time_published, author, description, title, complexity, themes, links, full_text

Уфф. Осталось применить её в цикле. Но давайте не будем торопиться: импортируем функцию `sleep` для задержки, чтобы на каждой итерации цикла, прежде чем перейти к следующей новости, Python ждал немного времени. Во-первых, это нужно, чтобы сайт «не понял», что мы его грабим, да ещё и автоматически. Во-вторых, с небольшой задержкой больше гарантии, что страница успеет загрузиться (сейчас это не очень важно, но станет актуально, когда мы будем обсуждать управление браузером через Selenium).

In [53]:
from time import sleep
from tqdm import tqdm

In [54]:
import random

In [55]:
urls

['https://nplus1.ru/news/2026/11/12/social-use-of-meow',
 'https://nplus1.ru/news/2026/11/13/alcedo-atthis-nestlings',
 'https://nplus1.ru/news/2026/11/13/chimpanzee-unvoiced-calls',
 'https://nplus1.ru/news/2026/11/13/brittle-solids-splitting',
 'https://nplus1.ru/news/2026/11/13/wildfire-smoke-asthma',
 'https://nplus1.ru/news/2026/11/13/omalizumab-alpha-gal',
 'https://nplus1.ru/news/2026/11/13/intellectual-disability-comorbidities',
 'https://nplus1.ru/news/2026/11/13/diabetic-foot',
 'https://nplus1.ru/news/2026/11/13/public-interest-in-dinosaurs',
 'https://nplus1.ru/news/2026/11/12/alphagenome2',
 'https://nplus1.ru/news/2026/11/12/stress-in-wastewater',
 'https://nplus1.ru/news/2026/11/12/social-use-of-meow',
 'https://nplus1.ru/news/2026/11/12/antihypertensives-perioperative-hypotension',
 'https://nplus1.ru/news/2026/11/12/standing-or-sitting',
 'https://nplus1.ru/news/2026/11/12/cluster-burnout',
 'https://nplus1.ru/news/2026/11/12/fluoridated-and-cognitive',
 'https://nplus

In [56]:
news = [] # это будет список кортежей, в которых будут храниться данные по каждой новости

for link in tqdm(urls):
    try:
        res = GetNews(link)
        news.append(res)
        sleep(random.random()) # случайная задержка до 1 секунды
    except:
        print('Ссылка битая')

100%|██████████| 48/48 [01:34<00:00,  1.97s/it]


Разберёмся, почему одна из новостей не обработалась. Возьмём её ссылку и повторим шаги функции вручную (индекс у вас может быть другим):

In [57]:
urls[40]

'https://nplus1.ru/news/2026/10/29/double-subduction-and-rifting'

In [58]:
page0 = requests.get(urls[40])
soup0 = BeautifulSoup(page0.text, 'lxml')

In [59]:
paragraphs = soup0.find_all('p', {'class': 'mb-6'})[1:]

In [60]:
links = set()
for p in paragraphs:
    links |= {i.get('href') for i in p.find_all('a')}

In [37]:
links - {None}

{'http://www.science.org/doi/10.1126/science.aef5187',
 'https://adamfholt.github.io/documents/papers/holt_et_al_epsl2018.pdf',
 'https://agupubs.onlinelibrary.wiley.com/doi/10.1029/2003TC001621',
 'https://karpinskyinstitute.ru/ru/public/sprav/geodictionary/article.php?ELEMENT_ID=83505',
 'https://nplus1.ru/news/2023/10/10/pontus-plate',
 'https://nplus1.ru/news/2026/06/01/emergence-of-euphrates-river',
 'https://pubs.geoscienceworld.org/gsa/geology/article/46/8/695/543292',
 'https://www.researchgate.net/profile/Paul-Umhoefer-2/publication/274990405_Why_did_the_Southern_Gulf_of_California_rupture_so_rapidly-Oblique_divergence_across_hot_weak_lithosphere_along_a_tectonically_active_margin/links/594855360f7e9b1d9b233192/Why-did-the-Southern-Gulf-of-California-rupture-so-rapidly-Oblique-divergence-across-hot-weak-lithosphere-along-a-tectonically-active-margin.pdf',
 'https://www.researchgate.net/publication/321045219_CENOZOIC_EXTENSION_AND_MAGMATISM_IN_WESTERN_MEXICO_LINKING_THE_SIERRA_

Так теперь выглядит первый элемент списка:

In [38]:
news[0]

('https://nplus1.ru/news/2026/11/12/social-use-of-meow',
 '12.11.26',
 '13:07',
 'Катерина Петрова',
 'Кошки замяукали в\xa0присутствии хозяина и\xa0при виде недоступной еды\nВ\xa0одиночестве они мяукали реже',
 'Кошки замяукали в\xa0присутствии хозяина и\xa0при виде недоступной еды',
 '1.5',
 'Зоология',
 'https://www.sciencedirect.com/science/article/abs/pii/S0003347226002940, https://psycnet.apa.org/doiLanding?doi=10.1037%2F0735-7036.118.3.287, https://nplus1.ru/news/2022/10/28/talking-with-cats',
 'Бразильские ученые обнаружили, что мяуканье домашних кошек скорее всего адресовано человеку. Когда перед кошками ставили прозрачный контейнер с едой, они мяукали чаще, если хозяин был в комнате, чем когда они были одни. Более чем в половине случаев питомцы смотрели на хозяина и часто переводили взгляд с него на контейнер. Исследование опубликовано в Animal Behaviour.\nДомашние кошки мяукают чаще диких, и обычно мяуканье происходит при общении с людьми, а не с другими кошками. Из этих наб

Импортируем `pandas` и создадим датафрейм из списка кортежей:

In [39]:
import pandas as pd

In [40]:
df = pd.DataFrame(news)

In [41]:
df.head(2)

,0,1,2,3,4,5,6,7,8,9
0,https://nplus1.ru/news/2026/11/12/social-use-o...,12.11.26,13:07,Катерина Петрова,Кошки замяукали в присутствии хозяина и при ви...,Кошки замяукали в присутствии хозяина и при ви...,1.5,Зоология,https://www.sciencedirect.com/science/article/...,"Бразильские ученые обнаружили, что мяуканье до..."
1,https://nplus1.ru/news/2026/11/13/alcedo-atthi...,13.11.26,19:12,Сергей Коленов,Двухнедельные птенцы зимородков на 41 процент ...,Двухнедельные птенцы зимородков на 41 процент ...,2.7,Зоология,https://nplus1.ru/news/2022/01/15/mynas-vs-kin...,Обыкновенные зимородки в первые дни жизни очен...


Переименуем столбцы в датафрейме.

In [42]:
df.columns = ['url', 'date_published', 'time_published', 'author', 'description', 'title', 'complexity', 'themes', 'links', 'full_text']

Теперь внесём изменения: сделаем столбец `complexity` числовым — типа *float*.

In [43]:
df['complexity'] = df['complexity'].apply(float)

Теперь сложность хранится в датафрейме как количественный показатель, и описывать её можно соответствующим образом:

In [44]:
df.complexity.describe()

count    48.000000
mean      3.206250
std       1.416229
min       1.300000
25%       2.200000
50%       2.650000
75%       3.900000
max       7.100000
Name: complexity, dtype: float64

In [45]:
df.to_csv('news_plus_1.csv',
          index = False,
          sep = ';', # сюда можно поставить произвольный символ, чтобы не ломать текст новости
          encoding = 'utf-8-sig')